# Hyperliquid vault of vaults - Sharpe entry gate grid search (daily rebalance)

- Variant of notebook 37 (`37-hyperliquid-sharpe-entry-gate-dynamic-universe`)
- Changed rebalance cycle from 4-day to **daily** (`CycleDuration.cycle_1d`)
- Uses dynamic Hyperliquid vault universe from `build_hyperliquid_vault_universe()`
- Min TVL $10k for increased early-period vault availability
- Grid searches: `rolling_returns_bars`, `weighting_method`, `max_concentration`, `waterfall`, `min_rolling_sharpe_entry`
- Uses optimiser objective function `optimise_profit`


# Set up

Set up Trading Strategy data client.


In [ ]:
import logging

from tradingstrategy.client import Client
from tradeexecutor.utils.notebook import setup_charting_and_output, OutputMode, set_notebook_logging

client = Client.create_jupyter_client()

# Set up drawing charts in interactive vector output mode.
# This is slower. See the alternative commented option below.
# setup_charting_and_output(OutputMode.interactive)

# Set up rendering static PNG images.
# This is much faster but disables zoom on any chart.
setup_charting_and_output(OutputMode.static, image_format="png", width=1500, height=1000)


logger = logging.getLogger("strategy")



## Chain configuration

- Choose target chains and their vults

In [ ]:
from eth_defi.token import USDC_NATIVE_TOKEN
from eth_defi.token import USDT_NATIVE_TOKEN
from eth_defi.token import WRAPPED_NATIVE_TOKEN

from tradingstrategy.chain import ChainId
from tradingstrategy.lending import LendingProtocolType

CHAIN_ID = ChainId.cross_chain
PRIMARY_CHAIN_ID = ChainId.ethereum
HYPERCORE_CHAIN_ID = ChainId.hypercore


# We define our main trading universe,
# and then Ethereum mainnet as a validation set
match CHAIN_ID:

    case ChainId.cross_chain:
        # Special backtest for pairs across all chains what a cross chain strategy would look lijke

        EXCHANGES = ("uniswap-v2", "uniswap-v3")
        SUPPORTING_PAIRS = [
            (ChainId.arbitrum, "uniswap-v3", "WETH", "USDC", 0.0005),
            (ChainId.ethereum, "uniswap-v3", "WETH", "USDC", 0.0005),
            (ChainId.ethereum, "uniswap-v3", "WBTC", "USDC", 0.003),
        ]
        LENDING_RESERVES = None
        PREFERRED_STABLECOIN = USDC_NATIVE_TOKEN[PRIMARY_CHAIN_ID].lower()


        # Dynamic Hypercore vault universe - fetched from Trading Strategy API
        # and cached per notebook in ~/.cache/indicators/
        # Clear cache with /clear-backtesting-cache skill
        from getting_started.hyperliquid_vault_universe import build_hyperliquid_vault_universe
        VAULTS = build_hyperliquid_vault_universe(
            notebook_id="38-hyperliquid-sharpe-entry-gate-daily-rebalance",
            min_tvl=10_000,
            top_n=120,
            min_age=0.15,
        )

        # Exclude Euro vaults, etc.
        # ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC", "USDT", "USDC.e"}
        ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC", "USDT", "USDC.e", "crvUSD", "USD₮0", "USDt", "USDT0", "USDS"}

        BENCHMARK_PAIRS = [
            (ChainId.ethereum, "uniswap-v3", "WBTC", "USDC", 0.003),
        ]
    case ChainId.arbitrum:

        EXCHANGES = ("uniswap-v2", "uniswap-v3")
        SUPPORTING_PAIRS = [
            (ChainId.arbitrum, "uniswap-v3", "WETH", "USDC", 0.0005),
        ]
        LENDING_RESERVES = None
        PREFERRED_STABLECOIN = USDC_NATIVE_TOKEN[PRIMARY_CHAIN_ID].lower()

        VAULTS = [
            # Arbitrum only
            (ChainId.arbitrum, "0x58bfc95a864e18e8f3041d2fcd3418f48393fe6a"),  # Plutus Hedge Token
            (ChainId.arbitrum, "0x959f3807f0aa7921e18c78b00b2819ba91e52fef"),  # gmUSDC
            (ChainId.arbitrum, "0xd3443ee1e91af28e5fb858fbd0d72a63ba8046e0"),  # gTrade (Gains) USDC
            (ChainId.arbitrum, "0x75288264fdfea8ce68e6d852696ab1ce2f3e5004"),  # Hype++
            (ChainId.arbitrum, "0x4b6f1c9e5d470b97181786b26da0d0945a7cf027"),  # Hypertrim USDC
            (ChainId.arbitrum, "0x0b2b2b2076d95dda7817e785989fe353fe955ef9"),  # Staked USDai
            (ChainId.arbitrum, "0x64ca76e2525fc6ab2179300c15e343d73e42f958"),  # Clearstar high yielsd USDC
            (ChainId.arbitrum, "0x7e97fa6893871a2751b5fe961978dccb2c201e65"),  # Gauntlet
            (ChainId.arbitrum, "0x1a996cb54bb95462040408c06122d45d6cdb6096"),  # Fluid
            (ChainId.arbitrum, "0xa91267a25939b2b0f046013fbf9597008f7f014b"),  # IPOR USDC Arbirum optimise
            (ChainId.arbitrum, "0x05d28a86e057364f6ad1a88944297e58fc6160b3"),  # Euler Arbitrum Yield USDC
            (ChainId.arbitrum, "0xc8248953429d707c6a2815653eca89846ffaa63b"),  # Curve LLAMMA asdCRV / crvUSD
            (ChainId.arbitrum, "0xf63b7f49b4f5dc5d0e7e583cfd79dc64e646320c"),  # Auto finance Tokemak ARB/USDC
            (ChainId.arbitrum, "0xeeaf2ccb73a01deb38eca2947d963d64cfde6a32"),  # Curve LLAMMA CRV / crvUSD
            (ChainId.arbitrum, "0xe5d6eb448ac5a762c1ebe8cd1692b9cd08025176"),  # DAMM stablecoin fund
        ]

        # Exclude Euro vaults, etc.
        # ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC", "USDT", "USDC.e"}
        ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC"}
    
    case ChainId.base:
        EXCHANGES = ("uniswap-v2", "uniswap-v3", "aerodrome")
        SUPPORTING_PAIRS = [
            (ChainId.base, "uniswap-v3", "WETH", "USDC", 0.0005),
        ]
        LENDING_RESERVES = None
        PREFERRED_STABLECOIN = USDC_NATIVE_TOKEN[ChainId.base].lower()

        VAULTS = [
            (ChainId.base, "0x13cd7cf42ccbaca8cd97e7f09572b6ea0de1097b"), # USDC-TIBBIR shares
        ]

        ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC"}


print(f"Chain universe: {CHAIN_ID}")
print(f"Vaults count: {len(VAULTS)}")
print(f"Allowed denomination tokens: {ALLOWED_VAULT_DENOMINATION_TOKENS}")

## Parameters

- Collection of parameters used in the calculations

In [ ]:
import datetime

import pandas as pd


from tradingstrategy.timebucket import TimeBucket
from tradeexecutor.strategy.cycle import CycleDuration
from tradeexecutor.strategy.parameters import StrategyParameters
from skopt.space import Categorical
from tradeexecutor.strategy.default_routing_options import TradeRouting

from tradeexecutor.utils.jupyter_notebook_name import get_notebook_id


class Parameters:

    id = get_notebook_id(globals())

    # We trade 1h candle
    candle_time_bucket = TimeBucket.d1
    cycle_duration = CycleDuration.cycle_1d
    
    chain_id = CHAIN_ID
    primary_chain_id = PRIMARY_CHAIN_ID
    exchanges = EXCHANGES
    
    #
    # Basket size, risk and balancing parameters.
    # Top 4 most important parameters from NB33 kept as Categorical.
    # Bottom 3 locked to best values from NB33.
    #   
    min_asset_universe = 5  # How many assets we need in the asset universe to start running the index
    max_assets_in_portfolio = 20  # Locked (5.7% importance in NB33)
    allocation = 0.95  # Allocate all cash to volatile pairs
    individual_rebalance_min_threshold_usd = 50.0  # Don't make buys less than this amount
    sell_rebalance_min_threshold = 10.0
    sell_threshold = 0.05  # Sell if asset is more than 5% of the portfolio
    per_position_cap_of_pool = 0.10  # Never own more than % of the lit liquidity of the trading pool
    max_concentration = Categorical([0.05, 0.10, 0.15, 0.20, 0.33])
    min_portfolio_weight = 0.0050  # Close position / do not open if weight is less than 50 BPS

    # Because backtest does not have enough data at the early universe,
    # we tweak it to simulate taking more concentrated positions in the past to 
    # avoid unused cash
    early_allocation_vault_count_threshold = 20
    early_allocation_max_contration = 15

    # Weighting and signal parameters.
    # rolling_returns_bars and weighting_method are the top 2 most important.
    rolling_returns_bars = Categorical([30, 60, 120, 240, 480])
    weighting_method = Categorical(["rolling_returns", "rolling_sharpe", "rolling_sortino", "inverse_volatility"])
    weight_function = "weight_1_slash_n"  # Locked (10.8% importance in NB33)
    waterfall = Categorical([True, False])
    volatility_window = 60  # Locked (10.5% importance in NB33)
    min_tvl = 10_000  # Minimum TVL in the vault before it can be considered investable
    max_rolling_volatility = 1.50  # 150% annualised — skip vault if vol exceeds this and TVL < $1M

    #
    # Sharpe entry gate — the single grid search parameter.
    # Minimum rolling Sharpe ratio required to enter a vault position.
    # Applied dynamically at each rebalance cycle.
    # None = filter disabled.
    # Based on notebook 35 findings: 70.5% feature importance.
    #
    min_rolling_sharpe_entry = Categorical([None, -1.0, 0.0, 0.5, 1.0])

    #     
    #
    # Backtesting only
    # Limiting factor: Not enough Hyperliquid vaults to trade
    #
    backtest_start = datetime.datetime(2025, 3, 1)
    backtest_end = datetime.datetime(2026, 3, 11)
    initial_cash = 100_000

    #
    # Live only
    #
    routing = TradeRouting.default
    required_history_period = datetime.timedelta(days=60*2)
    slippage_tolerance = 0.0060  # 0.6% 
    assummed_liquidity_when_data_missings = 10_000
    

parameters = StrategyParameters.from_class(Parameters)  # Convert to AttributedDict to easier typing with dot notation

from tradeexecutor.strategy.parameters import display_parameters
display_parameters(parameters)


# Trading pairs and market data

- This creates the strategy universe containing pair metadata and their prices
- The universe is "masked" by simply selecting pairs on the predefined pairs list

In [ ]:

from pathlib import Path
from typing import Callable
from tradingstrategy.pair import PandasPairUniverse


from eth_defi.vault.vaultdb import DEFAULT_VAULT_DATABASE, DEFAULT_RAW_PRICE_DATABASE


from tradingstrategy.utils.token_filter import filter_for_selected_pairs
from tradingstrategy.alternative_data.vault import load_vault_database


from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse
from tradeexecutor.strategy.execution_context import ExecutionContext, notebook_execution_context
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse, load_partial_data
from tradeexecutor.strategy.execution_context import ExecutionContext, notebook_execution_context
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.analysis.pair import display_strategy_universe
from tradeexecutor.strategy.pandas_trader.trading_universe_input import CreateTradingUniverseInput
from tradeexecutor.analysis.vault import display_vaults



# Hack to support vault data exposure to live trading universe creation
from dotenv import load_dotenv
load_dotenv(override=True)  # Loads variables from .env file


def create_trading_universe(
    input: CreateTradingUniverseInput,
) -> TradingStrategyUniverse:
    """Create the trading universe.

    - Load Trading Strategy full pairs dataset

    - Load built-in Coingecko top 1000 dataset

    - Get all DEX tokens for a certain Coigecko category

    - Load OHCLV data for these pairs

    - Load also BTC and ETH price data to be used as a benchmark
    """

    execution_context = input.execution_context
    client = input.client
    timestamp = input.timestamp
    parameters = input.parameters or Parameters  # Some CLI commands do not support yet passing this
    universe_options = input.universe_options

    if execution_context.live_trading:
        # Live trading, send strategy universe formation details
        # to logs
        debug_printer = logger.info
    else:
        # Jupyter notebook inline output
        debug_printer = print

    chain_id = parameters.primary_chain_id

    debug_printer(f"Preparing trading universe on chain {chain_id.get_name()}")

    # Pull out our benchmark pairs ids.
    # We need to construct pair universe object for the symbolic lookup.
    # TODO: PandasPairUniverse(buidl_index=True) - speed this up by skipping index building
    all_pairs_df = client.fetch_pair_universe().to_pandas()
    pairs_df= filter_for_selected_pairs(
        all_pairs_df,
        SUPPORTING_PAIRS,
    )    

    debug_printer(f"We have total {len(all_pairs_df)} pairs in dataset and going to use {len(pairs_df)} pairs for the strategy")

    # Check which vaults we can include based on allowed deposit tokens for this backtest
    # Use local vault database (~/.tradingstrategy/vaults/) instead of
    # the bundled one, as the bundled one may not have all chains
    vault_universe = load_vault_database(path=DEFAULT_VAULT_DATABASE)
    total_vaults = vault_universe.get_vault_count()
    vault_universe = vault_universe.limit_to_vaults(VAULTS, check_all_vaults_found=False)
    vault_universe = vault_universe.limit_to_denomination(ALLOWED_VAULT_DENOMINATION_TOKENS, check_all_vaults_found=True)
    debug_printer(f"Loaded total {vault_universe.get_vault_count()} vaults from the total of {total_vaults} in vault database, source vaults count: {len(VAULTS)}")

    # Default vault data bundle path for backtesting
    vault_bundled_price_data = DEFAULT_RAW_PRICE_DATABASE
    debug_printer(f"Using vault price data for backtesting from {vault_bundled_price_data}")

    dataset = load_partial_data(
        client=client,
        time_bucket=parameters.candle_time_bucket,
        pairs=pairs_df,
        execution_context=execution_context,
        universe_options=universe_options,
        liquidity=True,
        liquidity_time_bucket=TimeBucket.d1,
        lending_reserves=LENDING_RESERVES,
        vaults=vault_universe,
        vault_bundled_price_data=vault_bundled_price_data,
        check_all_vaults_found=True,
    )

    debug_printer("Creating strategy universe with price feeds and vaults")
    strategy_universe = TradingStrategyUniverse.create_from_dataset(
        dataset,
        reserve_asset=PREFERRED_STABLECOIN,
        forward_fill=True,  # We got very gappy data from low liquid DEX coins
        forward_fill_until=timestamp,
        primary_chain=parameters.primary_chain_id
    )
    return strategy_universe


universe_input = CreateTradingUniverseInput(
    execution_context=notebook_execution_context,
    client=client,
    timestamp=None,
    parameters=parameters,
    universe_options=UniverseOptions.from_strategy_parameters_class(Parameters, notebook_execution_context),
    execution_model=None,
)

strategy_universe = create_trading_universe(universe_input)


print("Universe creation done")

df = display_strategy_universe(
    strategy_universe, 
    sort_key="base",
    sort_numeric=False,
    limit=75,
    show_token_risk=True,
    show_tokensniffer=True,
)

df = df.head(3)

from tradeexecutor.utils.notebook import display_dataframe_with_html

display_dataframe_with_html(df)

# Indicators

- Precalculate indicators used by the strategy

In [ ]:
import pandas as pd
from IPython.display import HTML
import pandas_ta

from tradeexecutor.strategy.pandas_trader.indicator import IndicatorSource
from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse
from tradeexecutor.strategy.pandas_trader.indicator import calculate_and_load_indicators_inline
from tradeexecutor.strategy.pandas_trader.indicator import IndicatorDependencyResolver
from tradeexecutor.state.types import USDollarAmount
from tradeexecutor.strategy.pandas_trader.indicator_decorator import IndicatorRegistry
from tradeexecutor.analysis.indicator import display_indicators
from tradeexecutor.state.identifier import TradingPairIdentifier


indicators = IndicatorRegistry()

empty_series = pd.Series([], index=pd.DatetimeIndex([]))


@indicators.define()
def rolling_returns(
    close: pd.Series,
    rolling_returns_bars: int = 60,
) -> pd.Series:
    """Calculate annualised rolling returns.

    - Use last 60 days of price data (max)
    - Require minimum 7 days of data
    - Annualise all returns to be comparable across different data lengths
    """
    min_periods = 7

    first_price = close.rolling(
        window=rolling_returns_bars,
        min_periods=min_periods,
    ).apply(lambda x: x[0], raw=True)

    actual_days = close.rolling(
        window=rolling_returns_bars,
        min_periods=min_periods,
    ).apply(lambda x: len(x), raw=True)

    period_return = close / first_price - 1
    annualised = (1 + period_return) ** (365 / actual_days) - 1
    return annualised



@indicators.define()
def rolling_volatility(
    close: pd.Series,
    volatility_window: int = 180,
) -> pd.Series:
    """Calculate annualised rolling volatility.

    - Use last 6 months (180 days) of daily price data
    - If fewer than 180 days available, annualise whatever exists
    - Require minimum 14 days of data
    """
    min_periods = 14
    daily_returns = close.pct_change()
    rolling_std = daily_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).std()
    annualised = rolling_std * (365 ** 0.5)
    return annualised


@indicators.define()
def rolling_sharpe(
    close: pd.Series,
    volatility_window: int = 180,
) -> pd.Series:
    """Calculate rolling Sharpe ratio.

    - Annualised return / annualised volatility
    - Uses volatility_window for lookback period
    - Requires minimum 14 days of data
    """
    min_periods = 14
    daily_returns = close.pct_change()

    rolling_mean = daily_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).mean()

    rolling_std = daily_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).std()

    # Annualise: mean * 365 / (std * sqrt(365)) = mean * sqrt(365) / std
    sharpe = (rolling_mean * 365) / (rolling_std * (365 ** 0.5))
    return sharpe


@indicators.define()
def rolling_sortino(
    close: pd.Series,
    volatility_window: int = 180,
) -> pd.Series:
    """Calculate rolling Sortino ratio.

    - Annualised return / annualised downside deviation
    - Uses volatility_window for lookback period
    - Requires minimum 14 days of data
    """
    min_periods = 14
    daily_returns = close.pct_change()

    rolling_mean = daily_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).mean()

    # Downside deviation: std of negative returns only (clipped at 0)
    downside_returns = daily_returns.clip(upper=0)
    rolling_downside_std = downside_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).std()

    # Annualise
    sortino = (rolling_mean * 365) / (rolling_downside_std * (365 ** 0.5))
    return sortino

@indicators.define(source=IndicatorSource.tvl)
def tvl(
    close: pd.Series,
    execution_context: ExecutionContext,
    timestamp: pd.Timestamp,
) -> pd.Series:
    """Get TVL series for a pair.

    - Because TVL data is 1d and we use 1h everywhere else, we need to forward fill

    - Use previous hourly close as the value
    """
    if execution_context.live_trading:
        # TVL is daily data.
        # We need to forward fill until the current hour.
        # Use our special ff function.        
        assert isinstance(timestamp, pd.Timestamp), f"Live trading needs forward-fill end time, we got {timestamp}"
        from tradingstrategy.utils.forward_fill import forward_fill
        df = pd.DataFrame({"close": close})
        df_ff = forward_fill(
            df,
            Parameters.candle_time_bucket.to_frequency(),
            columns=("close",),
            forward_fill_until=timestamp,
        )
        series = df_ff["close"]
        return series
    else:
        return close.resample("1h").ffill()




@indicators.define()
def drawdown_from_peak(
    close: pd.Series,
) -> pd.Series:
    """Calculate running drawdown from share price peak.

    Returns negative values, e.g. -0.20 means vault is 20% below ATH.
    """
    cummax = close.cummax()
    drawdown = close / cummax - 1
    return drawdown


@indicators.define(dependencies=(tvl,), source=IndicatorSource.dependencies_only_universe)
def tvl_inclusion_criteria(   
    min_tvl: USDollarAmount,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    """The pair must have min XX,XXX USD one-sided TVL to be included.

    - If the Uniswap pool does not have enough ETH or USDC deposited, skip the pair as a scam

    :return:
        Series where each timestamp is a list of pair ids meeting the criteria at that timestamp
    """
    
    series = dependency_resolver.get_indicator_data_pairs_combined(tvl)
    mask = series >= min_tvl
    # Turn to a series of lists
    mask_true_values_only = mask[mask == True]
    series = mask_true_values_only.groupby(level='timestamp').apply(lambda x: x.index.get_level_values('pair_id').tolist())
    return series



@indicators.define(
    source=IndicatorSource.strategy_universe
)
def trading_availability_criteria(
    strategy_universe: TradingStrategyUniverse,
) -> pd.Series:
    """Is pair tradeable at each hour.

    - The pair has a price candle at that
    - Mitigates very corner case issues that TVL/liquidity data is per-day whileas price data is natively per 1h
      and the strategy inclusion criteria may include pair too early hour based on TVL only,
      leading to a failed attempt to rebalance in a backtest
    - Only relevant for backtesting issues if we make an unlucky trade on the starting date
      of trading pair listing

    :return:
        Series with with index (timestamp) and values (list of pair ids trading at that hour)
    """
    # Trading pair availability is defined if there is a open candle in the index for it.
    # Because candle data is forward filled, we should not have any gaps in the index.
    candle_series = strategy_universe.data_universe.candles.df["open"]
    pairs_per_timestamp = candle_series.groupby(level='timestamp').apply(lambda x: x.index.get_level_values('pair_id').tolist())
    return pairs_per_timestamp


@indicators.define(
    dependencies=[
        tvl_inclusion_criteria,
        trading_availability_criteria
    ],
    source=IndicatorSource.strategy_universe
)
def inclusion_criteria(
    strategy_universe: TradingStrategyUniverse,
    min_tvl: USDollarAmount,
    dependency_resolver: IndicatorDependencyResolver
) -> pd.Series:
    """Pairs meeting all of our inclusion criteria.

    - Give the tradeable pair set for each timestamp

    :return:
        Series where index is timestamp and each cell is a list of pair ids matching our inclusion criteria at that moment
    """

    # Filter out benchmark pairs like WETH in the tradeable pair set
    benchmark_pair_ids = set(strategy_universe.get_pair_by_human_description(desc).internal_id for desc in SUPPORTING_PAIRS)

    tvl_series = dependency_resolver.get_indicator_data(
        tvl_inclusion_criteria,
        parameters={
            "min_tvl": min_tvl,
        },
    )

    trading_availability_series = dependency_resolver.get_indicator_data(trading_availability_criteria)

    #
    # Process all pair ids as a set and the final inclusion
    # criteria is union of all sub-criterias
    #

    df = pd.DataFrame({
        "tvl_pair_ids": tvl_series,
        "trading_availability_pair_ids": trading_availability_series,
    })

    # https://stackoverflow.com/questions/33199193/how-to-fill-dataframe-nan-values-with-empty-list-in-pandas
    df = df.fillna("").apply(list)

    def _combine_criteria(row):
        final_set = set(row["tvl_pair_ids"]) & \
                    set(row["trading_availability_pair_ids"])
        return final_set - benchmark_pair_ids

    union_criteria = df.apply(_combine_criteria, axis=1)

    # Inclusion criteria data can be spotty at the beginning when there is only 0 or 1 pairs trading,
    # so we need to fill gaps to 0
    full_index = pd.date_range(
        start=union_criteria.index.min(),
        end=union_criteria.index.max(),
        freq=Parameters.candle_time_bucket.to_frequency(),
    )
    reindexed = union_criteria.reindex(full_index, fill_value=[])
    return reindexed


@indicators.define(dependencies=(tvl_inclusion_criteria,), source=IndicatorSource.dependencies_only_universe)
def tvl_included_pair_count(
        min_tvl: USDollarAmount,
        dependency_resolver: IndicatorDependencyResolver
) -> pd.Series:
    """Calculate number of pairs in meeting volatility criteria on each timestamp"""
    series = dependency_resolver.get_indicator_data(
        tvl_inclusion_criteria,
        parameters={"min_tvl": min_tvl},
    )
    series = series.apply(len)

    # TVL data can be spotty at the beginning when there is only 0 or 1 pairs trading,
    # so we need to fill gaps to 0
    full_index = pd.date_range(
        start=series.index.min(),
        end=series.index.max(),
        freq=Parameters.candle_time_bucket.to_frequency(),
    )
    # Reindex and fill NaN with zeros
    reindexed = series.reindex(full_index, fill_value=0)
    return reindexed


@indicators.define(dependencies=(inclusion_criteria,), source=IndicatorSource.dependencies_only_universe)
def all_criteria_included_pair_count(
    min_tvl: USDollarAmount,
    dependency_resolver: IndicatorDependencyResolver
) -> pd.Series:
    """Series where each timestamp is the list of pairs meeting all inclusion criteria.

    :return:
        Series with pair count for each timestamp
    """
    series = dependency_resolver.get_indicator_data(
        "inclusion_criteria",
        parameters={
            "min_tvl": min_tvl, 
        },
    )
    return series.apply(len)


@indicators.define(source=IndicatorSource.strategy_universe)
def trading_pair_count(
    strategy_universe: TradingStrategyUniverse,
) -> pd.Series:
    """Get number of pairs that trade at each timestamp.

    - Pair must have had at least one candle before the timestamp to be included

    - Exclude benchmarks pairs we do not trade

    :return:
        Series with pair count for each timestamp
    """

    benchmark_pair_ids = {strategy_universe.get_pair_by_human_description(desc).internal_id for desc in SUPPORTING_PAIRS}

    # Get pair_id, timestamp -> timestamp, pair_id index
    series = strategy_universe.data_universe.candles.df["open"]    
    swap_index = series.index.swaplevel(0, 1)

    seen_pairs = set()
    seen_data = {}

    for timestamp, pair_id in swap_index:
        if pair_id in benchmark_pair_ids:
            continue
        seen_pairs.add(pair_id)
        seen_data [timestamp] = len(seen_pairs)

    series = pd.Series(seen_data.values(), index=list(seen_data.keys()))
    return series





display_indicators(indicators)



# Time range for backtest

- Choose the backtesting time range
- Start when we have enough assets (`Parameters.min_asset_universe`) in our asset universe to form the first basket

In [ ]:
backtest_start = Parameters.backtest_start
backtest_end = Parameters.backtest_end

print(f"Time range is {backtest_start} - {backtest_end}")


# Algorithm and backtest

- Run the backtest

In [ ]:
import numpy as np

from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.strategy.alpha_model import AlphaModel
from tradeexecutor.state.trade import TradeExecution
from tradeexecutor.strategy.pandas_trader.strategy_input import StrategyInput, IndicatorDataNotFoundWithinDataTolerance
from tradeexecutor.state.visualisation import PlotKind
from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.strategy.tvl_size_risk import USDTVLSizeRiskModel
from tradeexecutor.strategy.weighting import weight_by_1_slash_n, weight_by_1_slash_signal, weight_passthrouh, weight_equal, weight_by_log
from tradeexecutor.utils.dedent import dedent_any
from tradeexecutor.strategy.pandas_trader.yield_manager import YieldManager, YieldRuleset, YieldWeightingRule, YieldDecisionInput
from tradeexecutor.strategy.execution_context import ExecutionContext, ExecutionMode
from getting_started.curator import is_quarantined



def decide_trades(
    input: StrategyInput
) -> list[TradeExecution]:
    """For each strategy tick, generate the list of trades."""
    parameters = input.parameters
    position_manager = input.get_position_manager()
    state = input.state
    timestamp = input.timestamp
    indicators = input.indicators
    strategy_universe = input.strategy_universe

    portfolio = position_manager.get_current_portfolio()
    equity = portfolio.get_total_equity()
    
    # All gone, stop doing decisions
    if input.execution_context.mode == ExecutionMode.backtesting:
        if equity < parameters.initial_cash * 0.10:
            return []
            
    # Build signals for each pair
    alpha_model = AlphaModel(
        timestamp,
        close_position_weight_epsilon=parameters.min_portfolio_weight,  # 10 BPS is our min portfolio weight
    )

    tvl_included_pair_count = indicators.get_indicator_value(
        "tvl_included_pair_count",
    )

    # Get pairs included in this rebalance cycle.
    # This includes pair that have been pre-cleared in inclusion_criteria()
    # with volume, volatility and TVL filters
    included_pairs = indicators.get_indicator_value(
        "inclusion_criteria",
        na_conversion=False,
    )
    if included_pairs is None:
        included_pairs = []

    # Determine which indicator to use as the signal based on weighting_method
    weighting_method = parameters.weighting_method
    if weighting_method == "rolling_returns" or weighting_method == "inverse_volatility":
        signal_indicator_name = "rolling_returns"
    elif weighting_method == "rolling_sharpe":
        signal_indicator_name = "rolling_sharpe"
    elif weighting_method == "rolling_sortino":
        signal_indicator_name = "rolling_sortino"

    # Set signal for each pair
    signal_count = 0
    pair_volatilities = {}
    sharpe_filtered = 0

    for pair_id in included_pairs:
        pair = strategy_universe.get_pair_by_id(pair_id)

        if not state.is_good_pair(pair):
            # Tradeable flag set to False, etc.
            continue

        if is_quarantined(pair.pool_address, timestamp):
            continue

        pair_signal = indicators.get_indicator_value(signal_indicator_name, pair=pair)
        if pair_signal is None or pair_signal < 0:
            continue

        # Skip small vaults with excessive volatility
        pair_vol = indicators.get_indicator_value("rolling_volatility", pair=pair)
        pair_tvl = indicators.get_indicator_value("tvl", pair=pair)
        if pair_tvl is not None and pair_tvl < 1_000_000:
            if pair_vol is not None and pair_vol > parameters.max_rolling_volatility:
                continue

        # Dynamic filter: minimum rolling Sharpe at entry
        if parameters.min_rolling_sharpe_entry is not None:
            pair_sharpe = indicators.get_indicator_value("rolling_sharpe", pair=pair)
            if pair_sharpe is not None and pair_sharpe < parameters.min_rolling_sharpe_entry:
                sharpe_filtered += 1
                continue

        # For inverse volatility, also collect rolling volatility per pair
        if weighting_method == "inverse_volatility":
            if pair_vol is None or pair_vol <= 0:
                continue
            pair_volatilities[pair.internal_id] = pair_vol

        alpha_model.set_signal(
            pair,
            pair_signal,
        )

        # Diagnostics reporting
        signal_count += 1

    # Calculate how much dollar value we want each individual position to be on this strategy cycle,
    # based on our total available equity
    portfolio = position_manager.get_current_portfolio()
    equity = portfolio.get_total_equity()
    portfolio_target_value = equity * parameters.allocation

    # We are greedy and take as many positions needed until we run out of investable equity
    alpha_model.select_top_signals(
        count=999,
    )

    # Assign weights based on weighting method
    if weighting_method == "inverse_volatility":
        # Override signal values with volatility for inverse vol weighting
        # select_top_signals picks by rolling returns, assign_weights uses 1/volatility
        for pair_id, signal_obj in alpha_model.signals.items():
            if pair_id in pair_volatilities:
                signal_obj.signal = pair_volatilities[pair_id]
        alpha_model.assign_weights(method=weight_by_1_slash_signal)
    else:
        # Resolve weight function from parameter
        weight_func_map = {
            "weight_equal": weight_equal,
            "weight_1_slash_n": weight_by_1_slash_n,
        }
        weight_func = weight_func_map[parameters.weight_function]
        alpha_model.assign_weights(method=weight_func)

    #
    # Normalise weights and cap the positions
    #
    size_risk_model = USDTVLSizeRiskModel(
        pricing_model=input.pricing_model,
        per_position_cap=parameters.per_position_cap_of_pool,  # This is how much % by all pool TVL we can allocate for a position
        missing_tvl_placeholder_usd=0.0,  # Placeholder for missing TVL data until we get the data off the chain
    )

    # Use higher concentration when there are not enough pairs in the cycle
    # to avoid sitting on unused cash during early backtest periods
    if len(included_pairs) < parameters.early_allocation_vault_count_threshold:
        max_weight = parameters.early_allocation_max_contration / 100
    else:
        max_weight = float(parameters.max_concentration)

    alpha_model.normalise_weights(
        investable_equity=portfolio_target_value,
        size_risk_model=size_risk_model,
        max_weight=max_weight,
        max_positions=parameters.max_assets_in_portfolio,
        waterfall=parameters.waterfall,
    )

    # Load in old weight for each trading pair signal,
    # so we can calculate the adjustment trade size
    alpha_model.update_old_weights(
        state.portfolio,
        ignore_credit=False,
    )
    alpha_model.calculate_target_positions(position_manager)

    # Shift portfolio from current positions to target positions
    # determined by the alpha signals (momentum)

    # rebalance_threshold_usd = portfolio_target_value * parameters.min_rebalance_trade_threshold_pct
    rebalance_threshold_usd = parameters.individual_rebalance_min_threshold_usd

    assert rebalance_threshold_usd > 0.1, "Safety check tripped - something like wrong with strat code"
    trades = alpha_model.generate_rebalance_trades_and_triggers(
        position_manager,
        min_trade_threshold=rebalance_threshold_usd,  # Don't bother with trades under XXXX USD
        invidiual_rebalance_min_threshold=parameters.individual_rebalance_min_threshold_usd,
        sell_rebalance_min_threshold=parameters.sell_rebalance_min_threshold,
        execution_context=input.execution_context,
    )
        
    # Add verbal report about decision made/not made,
    # so it is much easier to diagnose live trade execution.
    # This will be readable in Discord/Telegram logging.
    if input.is_visualisation_enabled():
        try:
            top_signal = next(iter(alpha_model.get_signals_sorted_by_weight()))
            if top_signal.normalised_weight == 0:
                top_signal = None
        except StopIteration:
            top_signal = None

        rebalance_volume = sum(t.get_value() for t in trades)

        report = dedent_any(f"""
        Cycle: #{input.cycle}
        Rebalanced: {'👍' if alpha_model.is_rebalance_triggered() else '👎'}
        Open/about to open positions: {len(state.portfolio.open_positions)} 
        Max position value change: {alpha_model.max_position_adjust_usd:,.2f} USD
        Rebalance threshold: {alpha_model.position_adjust_threshold_usd:,.2f} USD
        Trades decided: {len(trades)}
        Pairs total: {strategy_universe.data_universe.pairs.get_count()}
        Pairs meeting inclusion criteria: {len(included_pairs)}
        Pairs meeting TVL inclusion criteria: {tvl_included_pair_count}        
        Signals created: {signal_count}
        Filtered by Sharpe gate: {sharpe_filtered}
        Total equity: {portfolio.get_total_equity():,.2f} USD
        Cash: {position_manager.get_current_cash():,.2f} USD
        Investable equity: {alpha_model.investable_equity:,.2f} USD
        Accepted investable equity: {alpha_model.accepted_investable_equity:,.2f} USD
        Allocated to signals: {alpha_model.get_allocated_value():,.2f} USD
        Discarted allocation because of lack of lit liquidity: {alpha_model.size_risk_discarded_value:,.2f} USD
        Rebalance volume: {rebalance_volume:,.2f} USD
        """)

        # Most volatility pair signal weight (normalised): {max_vol_signal.normalised_weight * 100 if max_vol_signal else '-'} % (got {max_vol_signal.position_size_risk.get_relative_capped_amount() * 100 if max_vol_signal else '-'} % of asked size)
        if top_signal:
            assert top_signal.position_size_risk
            report += dedent_any(f"""
            Top signal pair: {top_signal.pair.get_ticker()}
            Top signal value: {top_signal.signal}
            Top signal weight: {top_signal.raw_weight}
            Top signal weight (normalised): {top_signal.normalised_weight * 100:.2f} % (got {top_signal.position_size_risk.get_relative_capped_amount() * 100:.2f} % of asked size)
            """)

        for flag, count in alpha_model.get_flag_diagnostics_data().items():
            report += f"Signals with flag {flag.name}: {count}\n"

        state.visualisation.add_message(
            timestamp,
            report,
        )

        state.visualisation.set_discardable_data("alpha_model", alpha_model)

    return trades  # Return the list of trades we made in this cycle


# Optimiser

- Run optimiser

In [ ]:
import logging
from tradeexecutor.backtest.optimiser import perform_optimisation
from tradeexecutor.backtest.optimiser import prepare_optimiser_parameters
from tradeexecutor.backtest.optimiser import MinTradeCountFilter
from tradeexecutor.backtest.optimiser_functions import optimise_sharpe, optimise_sortino, optimise_profit

# How many Gaussian Process iterations we do
iterations = 16

# What do we optimise for
# search_func = BalancedSharpeAndMaxDrawdownOptimisationFunction(sharpe_weight=0.75, max_drawdown_weight=0.25)
search_func = optimise_profit

optimiser_result = perform_optimisation(
    iterations=iterations,
    search_func=search_func,
    decide_trades=decide_trades,
    strategy_universe=strategy_universe,
    parameters=prepare_optimiser_parameters(Parameters),  # Handle scikit-optimise search space
    create_indicators=indicators.create_indicators,
    result_filter=MinTradeCountFilter(50),
    timeout=70*60,    
    batch_size=5,
    # We are searching wacky parameter combinations and
    # some of them lead to buggy strategies,
    # by setting ignore_wallet_errors we just zero out buggy
    # strategies instead of crashing with an exception
    ignore_wallet_errors=True,  
    # Uncomment for diagnostics
    # log_level=logging.INFO,
    # max_workers=1,
)

print(f"Optimise completed, optimiser searched {optimiser_result.get_combination_count()} combinations, with {optimiser_result.get_cached_count()} results read directly from cache. and {optimiser_result.get_filtered_count()} filtered results.")
print("Backtests failed with exception:", optimiser_result.get_failed_count())

# Results

- Show the top results of all optimiser iterations in a single table

In [ ]:
from tradeexecutor.analysis.optimiser import analyse_optimiser_result
from tradeexecutor.analysis.grid_search import render_grid_search_result_table


filtered = [r for r in optimiser_result.results if r.filtered]
print(f"Filtering out {len(filtered)} results")

# Optimiser already filtered for min_positions_threshold when doing the optimiser run
df = analyse_optimiser_result(
    optimiser_result,
    max_search_results=300,
    drop_duplicates=False,
)
print(f"Showing the best {len(df)} results")

render_grid_search_result_table(df)
 

## Equity curves

- Visualise equity curves of all grid search results in a single chart

In [ ]:
from tradeexecutor.visual.grid_search_basic import visualise_grid_search_equity_curves
from tradeexecutor.analysis.multi_asset_benchmark import get_benchmark_data

# Automatically create BTC and ETH buy and hold benchmark if present
# in the trading universe
benchmark_indexes = get_benchmark_data(
    strategy_universe,
    cumulative_with_initial_cash=Parameters.initial_cash,
)

# Extract GridSearchResult instances from optimiser results
grid_search_results = [r.result for r in optimiser_result.results if not r.filtered]

fig = visualise_grid_search_equity_curves(
    grid_search_results,
    benchmark_indexes=benchmark_indexes,
    log_y=False,
    group_by="weighting_method",
    group_by_secondary="min_rolling_sharpe_entry",
)
fig.show()


# Parameter analysis

## Decision tree visualisation

In [ ]:
from tradeexecutor.analysis.grid_search_parameters import analyse_decision_tree

fig, tree = analyse_decision_tree(df, analysis_metric="CAGR")
fig.show()

## Feature importance analysis

Use a regression model to determine which parameters have the strongest influence.



In [ ]:
from tradeexecutor.analysis.grid_search_parameters import analyse_feature_importance

fig, importances = analyse_feature_importance(df, analysis_metric="CAGR")
fig.show()
display(importances)

## Heatmaps for parameter pairs

In [ ]:
from tradeexecutor.analysis.grid_search_parameters import analyse_parameter_pair_heatmaps

figs = analyse_parameter_pair_heatmaps(df, analysis_metric="CAGR")
for fig in figs:
    fig.show()

# The best candidate equity curve

In [ ]:
from tradeexecutor.visual.grid_search import visualise_single_grid_search_result_benchmark

# GridSearchResult instance that gave the best performance
best_pick = optimiser_result.results[0].result
state = best_pick.hydrate_state()

print(f"The best result found for {search_func} was {best_pick}")

fig = visualise_single_grid_search_result_benchmark(
    best_pick, 
    strategy_universe, 
    initial_cash=Parameters.initial_cash,
    log_y=True,
)
fig.show()

# Portfolio performance (best pick)

- Compare buy and hold against our best performance

In [ ]:
from tradeexecutor.analysis.multi_asset_benchmark import compare_strategy_backtest_to_multiple_assets

returns = best_pick.returns

df = compare_strategy_backtest_to_multiple_assets(
    state=state,
    strategy_universe=strategy_universe,
    returns=returns,
    display=True,
    asset_count=3,
)
display(df)

# Trade summary (best pick)

- Show statistics about the made trades

In [ ]:
summary = best_pick.summary
display(summary.to_dataframe())

# Trading pair performance breakdown

- Show breakdown of different pairs on the best result

In [ ]:
from tradeexecutor.analysis.multipair import analyse_multipair
from tradeexecutor.analysis.multipair import format_multipair_summary

multipair_summary = analyse_multipair(state, show_address=True)
display(format_multipair_summary(multipair_summary))

# Best positions

- Find positions that made most profit for the leading backtest

In [ ]:
profit_by_position = [(p.get_realised_profit_usd(), p) for p in state.portfolio.get_all_positions()]
profit_by_position.sort(key=lambda t: t[0] or -999_999_999, reverse=True)

data = []

for profit, p in profit_by_position[0:5]:
    data.append({
        "Profit USD": profit,
        "Pair": p.pair.get_ticker(),
        "Position id": p.position_id,
        "Opened": p.opened_at,
        "Closed": p.closed_at,
        "Duration": p.closed_at - p.opened_at if p.closed_at else "(still open)",
        "Open price": p.get_opening_price(),
        "Close price": p.get_closing_price(),
        "Trades": p.get_trade_count(),
        "Price gain %": (p.get_closing_price() - p.get_opening_price()) / p.get_opening_price() * 100,
        "Weight at open %": p.get_capital_tied_at_open_pct() * 100,
    })

df = pd.DataFrame(data)
display(df)

# Rolling sharpe

- The rolling sharpe ratio of the best pick

In [ ]:
import plotly.express as px

from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns
from tradeexecutor.visual.equity_curve import calculate_rolling_sharpe

rolling_sharpe = calculate_rolling_sharpe(
    returns,
    freq="D",
    periods=90,
)

fig = px.line(rolling_sharpe, title='Strategy rolling Sharpe (6 months)')
fig.update_layout(showlegend=False)
fig.update_yaxes(title="Sharpe")
fig.update_xaxes(title="Time")
fig.show()

# Backtesting chart rendering for the best strategy

- Charts from the best grid search result using chart renderer

In [ ]:
from tradeexecutor.strategy.chart.definition import ChartRegistry, ChartKind, ChartInput
from tradeexecutor.strategy.chart.renderer import ChartBacktestRenderingSetup
from tradeexecutor.strategy.chart.standard.weight import equity_curve_by_asset
from tradeexecutor.strategy.chart.standard.weight import equity_curve_by_chain
from tradeexecutor.strategy.chart.standard.weight import weight_allocation_statistics
from tradeexecutor.strategy.chart.standard.weight import volatile_weights_by_percent
from tradeexecutor.strategy.chart.standard.weight import volatile_and_non_volatile_percent
from tradeexecutor.strategy.chart.standard.profit_breakdown import trading_pair_breakdown
from tradeexecutor.strategy.chart.standard.interest import vault_statistics
from tradeexecutor.strategy.chart.standard.vault import all_vault_positions
from tradeexecutor.strategy.chart.standard.vault import vault_position_timeline
from tradeexecutor.strategy.chart.standard.trading_universe import available_trading_pairs
from tradeexecutor.strategy.pandas_trader.indicator import load_indicators_inline
from tradeexecutor.strategy.parameters import StrategyParameters
from plotly.graph_objects import Figure

# Merge full strategy parameters with best combination's searched values
# (same pattern the optimiser uses internally — see optimiser.py ObjectiveWrapper)
merged_parameters = StrategyParameters.from_class(Parameters)
merged_parameters.update(best_pick.combination.as_dict())

# Load cached indicators for the best strategy from disk
# (calculated during grid search, no recalculation needed)
indicator_data = load_indicators_inline(
    strategy_universe=strategy_universe,
    create_indicators=indicators.create_indicators,
    parameters=merged_parameters,
)


def trading_pair_breakdown_with_chain(input: ChartInput) -> pd.DataFrame:
    """Trading pair breakdown with chain column and address."""
    return trading_pair_breakdown(
        input,
        show_chain=True,
        show_address=True,
    )

def all_vault_positions_by_profit(input: ChartInput) -> pd.DataFrame:
    """Vault positions sorted by profit, with address."""
    return all_vault_positions(
        input,
        sort_by="Profit USD",
        sort_ascending=False,
        show_address=True,
    )


charts = ChartRegistry(default_benchmark_pairs=BENCHMARK_PAIRS)
charts.register(equity_curve_by_asset, ChartKind.state_all_pairs)
charts.register(equity_curve_by_chain, ChartKind.state_all_pairs)
charts.register(weight_allocation_statistics, ChartKind.state_all_pairs)
charts.register(volatile_weights_by_percent, ChartKind.state_all_pairs)
charts.register(volatile_and_non_volatile_percent, ChartKind.state_all_pairs)
charts.register(trading_pair_breakdown_with_chain, ChartKind.state_all_pairs)
charts.register(vault_statistics, ChartKind.state_all_pairs)
charts.register(all_vault_positions_by_profit, ChartKind.state_single_vault_pair)
charts.register(vault_position_timeline, ChartKind.state_single_vault_pair)

charts.register(available_trading_pairs, ChartKind.indicator_all_pairs)
chart_renderer = ChartBacktestRenderingSetup(
    registry=charts,
    state=state,
    backtest_start_at=backtest_start,
    backtest_end_at=backtest_end,
    strategy_input_indicators=indicator_data,
)

# Available pairs (for the best strategy)

- Number of pairs available to trade every month

In [ ]:
fig, df = chart_renderer.render(available_trading_pairs, with_dataframe=True)
fig.show()
display(df.tail(5))

# Trading pair breakdown

- Trade success for each trading pair

In [ ]:
df = chart_renderer.render(trading_pair_breakdown_with_chain)
display(df)

# Vault performance

- Analyse the performance of our vaults

## Vault statistics

- Calculate interest accrued on different vaults for the strategy

In [ ]:
df = chart_renderer.render(vault_statistics)
display(df)

# Rolling Sharpe

- The rolling Sharpe ratio of the best strategy

In [ ]:
import plotly.express as px

from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns
from tradeexecutor.visual.equity_curve import calculate_rolling_sharpe

equity_curve = calculate_equity_curve(state)
returns = calculate_returns(equity_curve)

rolling_sharpe = calculate_rolling_sharpe(
    returns,
    freq="D",
    periods=180,
)

fig = px.line(rolling_sharpe, title='Strategy rolling Sharpe (6 months)')
fig.update_layout(showlegend=False)
fig.update_yaxes(title="Sharpe")
fig.update_xaxes(title="Time")
fig.show()

# Asset weights

- What assets were allocated over time
- Do both proportional % and USD weights

## Portfolio equity curve breakdown by asset

- Where did we make the profit

In [ ]:
fig = chart_renderer.render(equity_curve_by_asset)
fig.show()

## Weight allocation statistics

In [ ]:
from tradeexecutor.utils.notebook import display_dataframe_with_html

stats = chart_renderer.render(weight_allocation_statistics)
display_dataframe_with_html(stats)